## read docs

In [50]:
from pathlib import Path

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
DOCUMENTS_DIR = DATA_DIR / "documents"

In [51]:
from pathlib import Path

docs = []

for file in Path(DOCUMENTS_DIR).glob("*.txt"):
    text = file.read_text(encoding="utf-8")

    docs.append({
        "source": file.name,
        "text": text
    })

print("Documents:", len(docs))

for d in docs:
    print(d["source"])

Documents: 6
arrhythmia.txt
hypertension.txt
hypoxemia.txt
icu_protocol.txt
oxygen_therapy.txt
tachycardia.txt


## Chunking

In [52]:
from textwrap import wrap

chunks = []

chunk_size = 300

for doc in docs:

    text_chunks = wrap(
        doc["text"],
        width=chunk_size,
        break_long_words=False,
        break_on_hyphens=False
    )

    for idx, chunk in enumerate(text_chunks):

        chunks.append(
            {
                "source": doc["source"],
                "chunk_id": idx,
                "text": chunk
            }
        )

print("Chunks:", len(chunks))

Chunks: 14


In [53]:
chunks[:3]

[{'source': 'hypertension.txt',
  'chunk_id': 0,
  'text': 'Hypertension, commonly known as high blood pressure, is a chronic medical condition in which the force exerted by blood against artery walls remains elevated over time. Persistent hypertension increases the risk of cardiovascular disease, stroke, kidney damage, and heart failure.  Blood pressure is'},
 {'source': 'hypertension.txt',
  'chunk_id': 1,
  'text': 'measured using systolic and diastolic values. Elevated readings over an extended period may indicate hypertension. Risk factors include obesity, smoking, excessive salt intake, physical inactivity, diabetes, chronic stress, and family history.  Many individuals with hypertension have no noticeable'},
 {'source': 'hypertension.txt',
  'chunk_id': 2,
  'text': 'symptoms, which is why the condition is often referred to as a silent disease. Regular monitoring is essential because complications may develop before symptoms become apparent.  Healthcare professionals manage hype

## Embeddings

In [54]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\reems\Desktop\patient-monitoring-capstone\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [55]:
texts = [c["text"] for c in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
)

embeddings.shape

(14, 384)

## Vector Store

In [56]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(embeddings)
)

faiss.write_index(
    index,
    "../data/vector_store.faiss"
)

print(index.ntotal)

14


## Dense Retrieval

In [57]:
def dense_search(
    query,
    top_k=3
):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(
            {
                "source": chunks[idx]["source"],
                "chunk_id": chunks[idx]["chunk_id"],
                "text": chunks[idx]["text"]
            }
    )

    return results

In [58]:
dense_search(
    "What are symptoms of hypoxemia?"
)

[{'source': 'hypoxemia.txt',
  'chunk_id': 0,
  'text': 'Hypoxemia refers to an abnormally low level of oxygen in the blood. It is commonly assessed using pulse oximetry, which measures oxygen saturation. Healthy individuals typically maintain oxygen saturation values between 95 and 100 percent. Values below this range may indicate impaired oxygen'},
 {'source': 'hypoxemia.txt',
  'chunk_id': 1,
  'text': 'delivery to tissues.  Mild hypoxemia may cause few symptoms, while more severe cases can lead to shortness of breath, confusion, rapid breathing, increased heart rate, and cyanosis. In critically ill patients, prolonged oxygen deficiency may affect vital organs including the brain, heart, and'},
 {'source': 'hypoxemia.txt',
  'chunk_id': 2,
  'text': 'kidneys.  Common causes include pneumonia, chronic obstructive pulmonary disease, pulmonary edema, asthma exacerbations, respiratory infections, and impaired gas exchange in the lungs. Patients recovering from major surgery may also req

## BM25

In [59]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [
    chunk["text"].lower().split()
    for chunk in chunks
]

bm25 = BM25Okapi(
    tokenized_chunks
)


def bm25_search(
    query,
    top_k=3
):
    scores = bm25.get_scores(
        query.lower().split()
    )

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for idx, score in ranked[:top_k]:
        results.append(
        {
            "source": chunks[idx]["source"],
            "chunk_id": chunks[idx]["chunk_id"],
            "text": chunks[idx]["text"]
        }
)

    return results


bm25_search(
    "What is tachycardia?"
)

[{'source': 'hypertension.txt',
  'chunk_id': 2,
  'text': 'symptoms, which is why the condition is often referred to as a silent disease. Regular monitoring is essential because complications may develop before symptoms become apparent.  Healthcare professionals manage hypertension through lifestyle modifications and medications. Recommendations often'},
 {'source': 'hypertension.txt',
  'chunk_id': 0,
  'text': 'Hypertension, commonly known as high blood pressure, is a chronic medical condition in which the force exerted by blood against artery walls remains elevated over time. Persistent hypertension increases the risk of cardiovascular disease, stroke, kidney damage, and heart failure.  Blood pressure is'},
 {'source': 'hypoxemia.txt',
  'chunk_id': 3,
  'text': 'saturation below 92 percent is considered clinically significant and may trigger further assessment. Values below 90 percent often require urgent medical attention and supplemental oxygen therapy.  Management focuses on tr

## Hybrid Search

In [60]:
def hybrid_search(
    query,
    top_k=5
):
    dense_results = dense_search(
        query,
        top_k
    )

    bm25_results = bm25_search(
        query,
        top_k
    )

    combined = []

    seen = set()

    for result in bm25_results + dense_results:

        key = (
            result["source"],
            result["text"]
        )

        if key not in seen:

            combined.append(result)
            seen.add(key)

    return combined[:top_k]


hybrid_search(
    "tachycardia"
)

[{'source': 'hypertension.txt',
  'chunk_id': 0,
  'text': 'Hypertension, commonly known as high blood pressure, is a chronic medical condition in which the force exerted by blood against artery walls remains elevated over time. Persistent hypertension increases the risk of cardiovascular disease, stroke, kidney damage, and heart failure.  Blood pressure is'},
 {'source': 'hypertension.txt',
  'chunk_id': 1,
  'text': 'measured using systolic and diastolic values. Elevated readings over an extended period may indicate hypertension. Risk factors include obesity, smoking, excessive salt intake, physical inactivity, diabetes, chronic stress, and family history.  Many individuals with hypertension have no noticeable'},
 {'source': 'hypertension.txt',
  'chunk_id': 2,
  'text': 'symptoms, which is why the condition is often referred to as a silent disease. Regular monitoring is essential because complications may develop before symptoms become apparent.  Healthcare professionals manage hype

## Reciprocal Rank Fusion (RRF)

In [61]:
def dense_search_with_indices(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    return indices[0]


def bm25_search_with_indices(query, top_k=5):

    scores = bm25.get_scores(
        query.lower().split()
    )

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [idx for idx, _ in ranked[:top_k]]



def rrf_search(query, top_k=5, k=10):

    dense_rank = dense_search_with_indices(
        query,
        top_k
    )

    bm25_rank = bm25_search_with_indices(
        query,
        top_k
    )

    scores = {}

    for rank, doc_id in enumerate(dense_rank):

        scores[doc_id] = (
            scores.get(doc_id, 0)
            + 1 / (k + rank + 1)
        )

    for rank, doc_id in enumerate(bm25_rank):

        scores[doc_id] = (
            scores.get(doc_id, 0)
            + 1 / (k + rank + 1)
        )

    ranked_docs = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for doc_id, score in ranked_docs[:top_k]:

        results.append(
            {
                "score": score,
                "source": chunks[doc_id]["source"],
                "text": chunks[doc_id]["text"]
            }
        )

    return results


rrf_search(
    "What is tachycardia?"
)

[{'score': 0.16233766233766234,
  'source': 'hypertension.txt',
  'text': 'symptoms, which is why the condition is often referred to as a silent disease. Regular monitoring is essential because complications may develop before symptoms become apparent.  Healthcare professionals manage hypertension through lifestyle modifications and medications. Recommendations often'},
 {'score': 0.16025641025641024,
  'source': 'hypertension.txt',
  'text': 'Hypertension, commonly known as high blood pressure, is a chronic medical condition in which the force exerted by blood against artery walls remains elevated over time. Persistent hypertension increases the risk of cardiovascular disease, stroke, kidney damage, and heart failure.  Blood pressure is'},
 {'score': 0.1380952380952381,
  'source': 'hypoxemia.txt',
  'text': 'Hypoxemia refers to an abnormally low level of oxygen in the blood. It is commonly assessed using pulse oximetry, which measures oxygen saturation. Healthy individuals typically 

## Cross Encoder Reranking

In [62]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)



def rerank(query, retrieved_docs):

    pairs = [
        (query, doc["text"])
        for doc in retrieved_docs
    ]

    scores = reranker.predict(
        pairs
    )

    ranked = sorted(
        zip(retrieved_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked



results = rrf_search(
    "What is tachycardia?"
)

reranked = rerank(
    "What is tachycardia?",
    results
)

reranked

c:\Users\reems\Desktop\patient-monitoring-capstone\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[({'score': 0.08333333333333333,
   'source': 'icu_protocol.txt',
   'text': 'Patients in ICU should maintain oxygen saturation above 94%. Heart rate above 100 bpm may indicate tachycardia.  Intensive Care Unit monitoring involves continuous observation of critically ill patients using advanced monitoring systems. Healthcare providers track vital signs such as heart rate,'},
  np.float32(-1.043818)),
 ({'score': 0.09090909090909091,
   'source': 'icu_protocol.txt',
   'text': 'other medical emergencies.  Alarm thresholds are often configured to notify staff when measurements exceed predefined limits. Persistent tachycardia, severe hypoxemia, or sudden blood pressure changes may require immediate intervention.  Modern monitoring systems generate large volumes of'},
  np.float32(-3.4363294)),
 ({'score': 0.16025641025641024,
   'source': 'hypertension.txt',
   'text': 'Hypertension, commonly known as high blood pressure, is a chronic medical condition in which the force exerted by blood 

## Answer Generation using FLAN-T5

In [63]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)



In [65]:

def rag_answer(query):

    retrieved = rrf_search(
        query,
        top_k=5
    )

    reranked = rerank(
        query,
        retrieved
    )

    top_docs = [
        doc
        for doc, score in reranked[:3]
    ]

    context = "\n".join(
        doc["text"]
        for doc in top_docs
    )

    prompt = f"""
You are a medical assistant.
Answer the question using ONLY the context provided.
Give a complete sentence and include the source information.

Context:
{context}

Question:
{query}

Answer:
"""

    response = generator(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.3
)

    return {
        "question": query,
        "answer": response[0]["generated_text"],
        "citations": list(
            set(
                doc["source"]
                for doc in top_docs
            )
        )
    }


rag_answer(
    "What vital signs are monitored in the ICU?"
)

{'question': 'What vital signs are monitored in the ICU?',
 'answer': 'heart rate, oxygen saturation, respiratory rate, blood pressure, and body temperature',
 'citations': ['icu_protocol.txt']}

##  gen Answer + Citations

In [66]:
def answer_with_citations(query):

    retrieved = rrf_search(
        query,
        top_k=5
    )

    reranked = rerank(
        query,
        retrieved
    )

    top_docs = [
        doc
        for doc, score in reranked[:3]
    ]

    answer_parts = []

    citations = []

    for doc in top_docs:

        answer_parts.append(
            doc["text"]
        )

        citations.append(
            doc["source"]
        )

    return {
        "question": query,
        "answer": " ".join(answer_parts),
        "citations": citations
    }




In [68]:

answer_with_citations(
    "What heart rate is considered normal?"
)

{'question': 'What heart rate is considered normal?',
 'answer': 'Patients in ICU should maintain oxygen saturation above 94%. Heart rate above 100 bpm may indicate tachycardia.  Intensive Care Unit monitoring involves continuous observation of critically ill patients using advanced monitoring systems. Healthcare providers track vital signs such as heart rate, oxygen saturation, respiratory rate, blood pressure, and body temperature.  Continuous monitoring helps clinicians identify physiological deterioration before severe complications develop. Changes in heart rate or oxygen saturation may indicate infection, respiratory failure, cardiac instability, or Hypoxemia refers to an abnormally low level of oxygen in the blood. It is commonly assessed using pulse oximetry, which measures oxygen saturation. Healthy individuals typically maintain oxygen saturation values between 95 and 100 percent. Values below this range may indicate impaired oxygen',
 'citations': ['icu_protocol.txt', 'icu_p